# K5 trained on T2, read on Nexar val, DoTA-CAP-dev, TAD and DAD

Rule: `core/docs/final/KIM_RUN.md` §13, written before any number. **Printed, decides nothing:** K5 on Nexar stays the main model. One seed: **PRELIMINARY (n = 1)**. `nexar_test` is never touched (§13 T4). Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (T2 inputs, K5, LaGoVAD reads) | no | 2-3 min |
| 4 | train K5 on T2, s2024, 1,740 steps -> `outputs/kim_t2/K5/s2024` | yes | about as long as K5 on Nexar |
| 5 | bake TAD and DAD with T2 statistics | no | 5-10 min |
| 6 | K5 (T2) on DoTA-CAP-dev, TAD, DAD | yes | 15 min |
| 7 | `nexar_val`: K5 (T2), K5 (Nexar), LaGoVAD | yes | 20-40 min |
| 8 | `core.tools.k5_t2_report` -> **paste `REPORTS/k5_t2_report.md` to Claude** | no | 2 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to Thesis-Final. If the runtime dies in step 4 the local checkpoint is lost and the run starts again with the same seed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip, ckpts/best.ckpt
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: T2 and DoTA-CAP A2 bakes, KNN, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5, LaGoVAD reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/k5_t2_report.py').is_file(), 'this upload has no k5_t2_report -- re-upload kat-vad/'
assert '## 13. K5 trained on T2' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §13 -- re-upload kat-vad/'
assert '--baseline' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --baseline -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, K5T2, LAG = constants.KIM_MAIN_ARM, constants.KIM_T2_ROW, constants.ZS_LAGOVAD_ARM
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
T2NAME   = constants.DADA_ORIGIN_DATASET
EPOCHS   = constants.KIM_T2_EPOCHS                 # 20
STEPS    = constants.NEXAR_STEP_BUDGET             # 1,740 = 20 x 87 on T2
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # §13 T4: nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE
TAD_STRIDE = constants.V2_TADZ_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
FINAL_CACHE = constants.CACHE_ROOT                 # Thesis-Final/cache

# Training: E3's T2 recipe (§13 T1, T2)
T2_DATA  = Path(DRIVE_V2) / 'data' / f'{T2NAME}_v2'
T2_KNN   = V2_CACHE / 'knn' / f'{T2NAME}_v2' / constants.KNN_CACHE_FILENAME
A2_T2    = V2_CACHE / 'v2' / 'A2_S' / T2NAME        # T2 statistics; also the stats dir of the new bakes
E3_CFG   = V2_OUT / 'v2_e3' / 'A2' / f's{SEED}' / 'config.yaml'
# The row this is compared with
KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
K5_RUN   = KIM / K5 / f's{SEED}'
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'    # K5 (Nexar)'s statistics, for its Nexar read
LAGOVAD  = constants.OUTPUT_ROOT / constants.ZS_LAGOVAD_OUTPUT_DIRNAME
BEST     = constants.CKPT_ROOT / 'best.ckpt'
# Eval inputs
A2_CAP   = V2_CACHE / 'v2' / 'A2_S' / 'DoTA_CAP_s3_from_s1'   # E3's bake, T2 statistics
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'
TAD_V2S  = V2_CACHE / 'video' / ENC / f'{TAD_NAME}_s{TAD_STRIDE}_squash'
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
DAD_NCC  = FINAL_CACHE / 'clip' / f'{DAD_NAME}_s{DAD_STRIDE}_ncc'
DAD_V2S  = FINAL_CACHE / 'video' / ENC / f'{DAD_NAME}_s{DAD_STRIDE}_squash'
TAD_BAKE = FINAL_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s{TAD_STRIDE}_t2'   # written in step 5
DAD_BAKE = FINAL_CACHE / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_t2'   # written in step 5
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
NEXAR_DEFS = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'
# Outputs
ROOT     = constants.OUTPUT_ROOT / constants.KIM_T2_OUTPUT_DIRNAME
RUN      = ROOT / K5 / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
EVAL     = ROOT / 'eval'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_t2')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = (('T2 data', T2_DATA), ('T2 KNN', T2_KNN), ('T2 A2 bake', A2_T2 / constants.V2_INPUT_MANIFEST_FILENAME),
          ('E3 A2 config', E3_CFG), ('K5 checkpoint', K5_RUN / 'checkpoint_last.pt'), ('K5 verified', K5_RUN / 'verified.json'),
          ('K5 config', K5_RUN / 'config.yaml'), ('Nexar A2 stats', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('best.ckpt', BEST), ('DoTA-CAP A2 bake', A2_CAP / constants.V2_INPUT_MANIFEST_FILENAME), ('DoTA s1', DOTA_S1),
          ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('TAD CLIP', TAD_NCC), ('TAD V2-S', TAD_V2S),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME), ('DAD CLIP', DAD_NCC), ('DAD V2-S', DAD_V2S),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U), ('Nexar defs', NEXAR_DEFS),
          ('LaGoVAD DoTA read', LAGOVAD / 'eval' / 'dota' / LAG), ('LaGoVAD TAD read', LAGOVAD / 'eval' / 'tad' / LAG),
          ('LaGoVAD DAD read', LAGOVAD / 'eval' / constants.DAD_OUTPUT_SET / LAG),
          ('K5 DoTA read', KIM / 'eval' / 'dota' / K5), ('K5 TAD read', KIM / 'eval' / 'tad' / K5),
          ('K5 DAD read', KIM / 'eval' / constants.DAD_OUTPUT_SET / K5))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | output {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl
python -c "import av, einops, faiss, sklearn; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 4, 6, 7 need one'

In [ ]:
import torch
import yaml

from core.config import Config                  # noqa: E402
from core.data.v2_splits import load_split      # noqa: E402
from core.tools.v5_stage1 import _flat          # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def stage(src, dst, ids=None):
    """A Drive file or dir (only ``ids`` + json/npz manifests if given) to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    if ids is None:
        if not dst.exists():
            tmp = dst.with_name(dst.name + '.partial')
            shutil.rmtree(tmp, ignore_errors=True)
            shutil.copytree(src, tmp)
            os.replace(tmp, dst)
    else:
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())


# Preflight on small files only (C39)
k5_cfg = _flat(yaml.safe_load((K5_RUN / 'config.yaml').read_text()))
assert k5_cfg['kim.enabled'] and k5_cfg['kim.prototypes'] and k5_cfg['kim.edge_mask'], 'K5_RUN is not K5'
assert k5_cfg['dvs.theta'] == 1.0 and k5_cfg['data.dataset'] == constants.NEXAR_DATASET, 'K5_RUN is not K5 on Nexar'
k5_steps = json.loads((K5_RUN / 'verified.json').read_text())['step']
assert is_verified(K5_RUN, k5_steps) and metrics_last(K5_RUN / 'metrics.jsonl')['global_step'] == k5_steps, 'K5: J10'
T2_SHA = manifest(A2_T2)['train_ids_sha1']
for name, d in (('T2', A2_T2), ('DoTA-CAP', A2_CAP), ('Nexar', A2_NEXAR)):
    assert (manifest(d)['crn'], manifest(d)['motion']) == (constants.V2_OFF, ENC), f'{name} bake: {manifest(d)}'
assert manifest(A2_CAP)['train_ids_sha1'] == T2_SHA, 'the DoTA-CAP bake does not carry T2 statistics'
assert manifest(A2_NEXAR)['train_ids_sha1'] != T2_SHA, 'the Nexar stats carry T2 statistics'
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS, (len(TAD_IDS), len(DAD_IDS))
for name, d, ids in (('DoTA-CAP bake', A2_CAP, CAP_IDS), ('TAD CLIP', TAD_NCC, TAD_IDS), ('TAD V2-S', TAD_V2S, TAD_IDS),
                     ('DAD CLIP', DAD_NCC, DAD_IDS), ('DAD V2-S', DAD_V2S, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5 step {k5_steps} | DoTA-CAP-dev {len(CAP_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')

## 4. Train K5 on T2, s2024 (§13 T1)

In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={T2NAME}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', *K5_SETS]
DEFAULTS = _flat(Config().to_dict())
KIM_KEYS = {s.split('=')[0] for s in K5_SETS}


def config_diff(base, got, allowed):
    """Keys that differ beyond ``allowed``; a key added after ``base`` trained may sit at its default."""
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k) or k in allowed:
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run():
    payload = torch.load(CKPT, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(RUN / 'metrics.jsonl')['global_step'] == STEPS, 'J10'
    assert any(k.startswith('kim.') for k in payload['model']), 'no KIM weights in the checkpoint'
    got = _flat(yaml.safe_load((RUN / 'config.yaml').read_text()))
    e3 = _flat(yaml.safe_load(E3_CFG.read_text()))
    diff = config_diff(e3, got, KIM_KEYS)
    assert not diff, f'K5 (T2) differs from E3 A2 beyond DVS off + KIM: {diff}'
    diff = config_diff(k5_cfg, got, {'data.dataset', 'train.num_epochs'})
    assert not diff, f'K5 (T2) differs from K5 (Nexar) beyond the corpus and its epochs: {diff}'
    print(f'K5 (T2): J10 {STEPS} | == E3 A2 + {sorted(KIM_KEYS)} | == K5 except data.dataset, train.num_epochs')


if is_verified(RUN, STEPS):
    print(f'K5 (T2) s{SEED}: verified on Drive -- not retrained')
else:
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'run'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)],
         '--data-dir', stage(T2_DATA, LOCAL / 'stage' / 't2_data'), '--clip-dir', stage(A2_T2, LOCAL / 'stage' / 't2_a2'),
         '--knn-cache', stage(T2_KNN, LOCAL / 'stage' / 'knn.npz'), '--output-dir', local, *resume], log=LOCAL / 'train.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'J10 locally: {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, RUN / name)
    flush_drive()
    assert ckpt_step(CKPT) == STEPS, f'Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': CKPT.stat().st_size}))
    publish(LOCAL / 'verified.json', RUN / 'verified.json')
    print(f'trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")} '
          f'| W_out {last.get("kim_w_out_norm")} | proto cos {last.get("kim_proto_cos")}')
check_run()

## 5. Bake TAD and DAD with T2 statistics (§13 T2)

In [ ]:
def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(ids) + '\n')
    return path


L_STATS = stage(A2_T2, LOCAL / 'stage' / 't2_stats', [])     # manifest + statistics only
for name, ids, x_dir, u_dir, dst in (('tad', TAD_IDS, TAD_NCC, TAD_V2S, TAD_BAKE), ('dad', DAD_IDS, DAD_NCC, DAD_V2S, DAD_BAKE)):
    man = dst / constants.V2_INPUT_MANIFEST_FILENAME
    if not (man.exists() and json.loads(man.read_text())['baked_ids'] == len(ids)):
        local = LOCAL / 'bake' / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.build_v2_inputs', 'apply', '--stats-dir', L_STATS,
             '--clip-dir', stage(x_dir, LOCAL / 'stage' / f'{name}_x', ids), '--video-dir', stage(u_dir, LOCAL / 'stage' / f'{name}_u', ids),
             '--ids-file', write_ids(LOCAL / 'ids' / f'{name}.txt', ids), '--stride', 1, '--out-dir', local], log=LOCAL / 'bake.log')
        publish(local, dst)
    m = manifest(dst)
    assert (m['crn'], m['motion'], m['train_ids_sha1'], m['baked_ids']) == (constants.V2_OFF, ENC, T2_SHA, len(ids)), f'{name}: {m}'
    print(f'{name}: {m["baked_ids"]} videos, T2 statistics')
flush_drive()

## 6. K5 (T2) on DoTA-CAP-dev, TAD, DAD

In [ ]:
def read_dota():
    out = EVAL / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT,
             '--input-dir', stage(A2_CAP, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
             '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
             '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / 'dota.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    print(f"DoTA-CAP-dev: macro {r['macro']['mean']:.4f}")


def read_evaluate(name, dataset, data_dir, bake, ids, out):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', CKPT, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


L_TAD = LOCAL / 'stage' / 'tad_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
read_dota()
read_evaluate('tad', TAD_NAME, L_TAD, TAD_BAKE, TAD_IDS, EVAL / 'tad' / K5 / f's{SEED}')
read_evaluate('dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), DAD_BAKE, DAD_IDS,
              EVAL / constants.DAD_OUTPUT_SET / K5 / f's{SEED}')
flush_drive()

## 7. `nexar_val`: three rows (§13 T4, T5). K5 (Nexar) is in-domain here; `--final` is never passed

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON  # noqa: E402

L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
L_DEFS = LOCAL / 'stage' / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in NEXAR_DEFS.glob('*.json'):         # the definitions; the same for all three rows
    shutil.copy2(f, L_DEFS / f.name)
L_NEXAR_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])
L_T2_STATS = LOCAL / 'stage' / 't2_stats'
ROWS = {
    LAG: (['--baseline', constants.ZS_LAGOVAD_RUN, BEST], None),   # first: the cheapest place to find a loader problem
    K5: (['--run', f's{SEED}', K5_RUN / 'checkpoint_last.pt'], L_NEXAR_STATS),
    K5T2: (['--run', f's{SEED}', CKPT], L_T2_STATS),
}
for row, (ckpt_args, stats) in ROWS.items():
    out = EVAL / 'nexar' / row
    if not ((out / READOUT_JSON).exists() and (out / ITEM_AUCS_JSON).exists()):
        local = LOCAL / 'eval' / 'nexar' / row
        shutil.rmtree(local, ignore_errors=True)
        extra = [] if stats is None else ['--stats-dir', stats, '--video-s1-dir', L_U]
        run([sys.executable, '-m', 'core.tools.nexar_eval', *ckpt_args, *extra, '--census', CENSUS, '--clip-s1-dir', L_X,
             '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--out-dir', local], log=LOCAL / 'nexar.log')
        publish(local, out)
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
    print(f"{row}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
          f"whole macro {r['whole']['macro']['mean']:.4f} | micro {r['whole']['micro']:.4f} | AP {r['whole']['micro_ap']:.4f}")
flush_drive()

## 8. The table -> **paste `REPORTS/k5_t2_report.md` to Claude**

In [ ]:
local = LOCAL / 'report'
run([sys.executable, '-m', 'core.tools.k5_t2_report', '--lagovad-root', LAGOVAD, '--kim-root', KIM, '--t2-root', ROOT,
     '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / 'k5_t2_report.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste REPORTS/k5_t2_report.md to Claude')